# Sentiment Stability Analysis

This notebook loads the production sentiment stability feature set and yearly market returns, merges them by year, and evaluates whether sentiment deviation is related to forward returns.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def get_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == 'notebooks' else cwd

BASE_DIR = get_project_root()
if str(BASE_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(BASE_DIR / 'src'))

STABILITY_PATH = BASE_DIR / 'features' / 'sentiment_stability.csv'
MARKET_PATH = BASE_DIR / 'features' / 'market_returns.csv'

print(f'Loading sentiment stability features from: {STABILITY_PATH}')
print(f'Loading market returns from: {MARKET_PATH}')

stability = pd.read_csv(STABILITY_PATH)
market_returns = pd.read_csv(MARKET_PATH)

stability['year'] = pd.to_numeric(stability['year'], errors='coerce').astype('Int64')
market_returns['year'] = pd.to_numeric(market_returns['year'], errors='coerce').astype('Int64')

stability = stability.sort_values(['company_id', 'year'], kind='mergesort').reset_index(drop=True)
market_returns = market_returns.sort_values('year', kind='mergesort').reset_index(drop=True)

print('stability rows:', len(stability))
print('market rows:', len(market_returns))
print('stability year range:', stability['year'].min(), '->', stability['year'].max())
print('market year range:', market_returns['year'].min(), '->', market_returns['year'].max())


In [ ]:
analysis_df = stability.merge(market_returns, on='year', how='left', validate='many_to_one')
analysis_df = analysis_df.dropna(subset=['sentiment_deviation', 'next_year_return']).copy()
analysis_df['deviation_bucket'] = pd.qcut(analysis_df['sentiment_deviation'], q=3, labels=['low', 'mid', 'high'], duplicates='drop')

print('merged rows:', len(analysis_df))
analysis_df.head()


In [ ]:
correlation = analysis_df['sentiment_deviation'].corr(analysis_df['next_year_return'])
bucket_analysis = (
    analysis_df.groupby('deviation_bucket', observed=False)['next_year_return']
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .sort_index()
)

low_cutoff = analysis_df['sentiment_deviation'].quantile(0.10)
high_cutoff = analysis_df['sentiment_deviation'].quantile(0.90)
extreme_quantiles = pd.concat(
    [
        analysis_df.loc[analysis_df['sentiment_deviation'] <= low_cutoff].assign(extreme_bucket='lowest_10pct'),
        analysis_df.loc[analysis_df['sentiment_deviation'] >= high_cutoff].assign(extreme_bucket='highest_10pct'),
    ],
    ignore_index=True,
)
extreme_summary = (
    extreme_quantiles.groupby('extreme_bucket')['next_year_return']
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .sort_index()
)

print('correlation (deviation vs forward returns):', correlation)
display(bucket_analysis)
display(extreme_summary)


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(analysis_df['sentiment_deviation'], analysis_df['next_year_return'], alpha=0.7)
plt.axhline(0.0, color='black', linewidth=1, linestyle='--')
plt.title('Sentiment Deviation vs Forward Returns')
plt.xlabel('Sentiment Deviation')
plt.ylabel('Next-Year Return')
plt.tight_layout()
plt.show()


In [ ]:
yearly_summary = (
    analysis_df.groupby('year', as_index=False)
    .agg(
        mean_sentiment_score=('sentiment_score', 'mean'),
        mean_sentiment_deviation=('sentiment_deviation', 'mean'),
        next_year_return=('next_year_return', 'mean'),
    )
)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(yearly_summary['year'], yearly_summary['mean_sentiment_score'], color='tab:blue', marker='o', label='Mean Sentiment Score')
ax1.plot(yearly_summary['year'], yearly_summary['mean_sentiment_deviation'], color='tab:orange', marker='s', label='Mean Sentiment Deviation')
ax1.set_xlabel('Year')
ax1.set_ylabel('Sentiment Metrics')

ax2 = ax1.twinx()
ax2.plot(yearly_summary['year'], yearly_summary['next_year_return'], color='tab:green', marker='^', label='Next-Year Return')
ax2.set_ylabel('Next-Year Return')

lines = ax1.get_lines() + ax2.get_lines()
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc='best')
plt.title('Sentiment Stability and Forward Returns Over Time')
plt.tight_layout()
plt.show()

yearly_summary.tail()
